In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Set professional styling
plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.titlesize": 10,
    "lines.linewidth": 1.5,
    "figure.autolayout": False, # Manual layout control for legend
})

# 1. Consistent Color & Name Mapping
# This ensures "Model Sensitivity" is the same color in both plots
COLOR_MAP = {
    "Data Sensitivity": "#3498db",   # A distinct, bright sky blue
    "Model Sensitivity": "#f1c40f",  # A vibrant, distinct golden yellow
    "Uncertainty": "#e74c3c",        # Soft Pastel Red
    "Residual-Reduction": "#2ecc71", # Soft Pastel Green
    "Passive Baseline": "#2f3640",   # Soft Light Grey
}

name_mapping = {
    "RANDOM": "Passive Baseline",
    "ONLINE_S": "Model Sensitivity",
    "OFFLINE_S": "Model Sensitivity",
    "EXPERIMENTAL_S": "Data Sensitivity",
    "ONLINE_RES": "Residual-Reduction",
    "OFFLINE_RES": "Residual-Reduction",
    "ONLINE_UNCERTAINTY": "Uncertainty",
    "OFFLINE_UNCERTAINTY": "Uncertainty",
}

online_keys = ["ONLINE_S", "ONLINE_RES", "ONLINE_UNCERTAINTY", "RANDOM"]
offline_keys = ["EXPERIMENTAL_S", "OFFLINE_S", "OFFLINE_RES", "OFFLINE_UNCERTAINTY", "RANDOM"]

def plot_data_on_ax(ax, data, keys, title):
    """Helper to plot a specific set of keys on a provided axis."""
    lines = []
    labels = []
    
    for k in keys:
        raw_values = data[k]
        label = name_mapping.get(k, k)
        
        # Statistics
        median = np.nanmedian(raw_values, axis=0)
        lower = np.nanpercentile(raw_values, 25, axis=0)
        upper = np.nanpercentile(raw_values, 75, axis=0)

        #print(k, median)

        color = COLOR_MAP.get(label, "#333333")
        
        # Style logic
        is_primary = "Sensitivity" in label
        alpha_val = 1.0 if is_primary else 0.4
        linewidth = 2.0 if is_primary else 1.2
        zorder = 10 if is_primary else 2
        alpha_cloud = 0.15 if is_primary else 0.05

        # Force ticks to be consistent across both plots
        ax.yaxis.set_major_formatter(ticker.LogFormatterExponent()) 
        # Or use ScalarFormatter if the range is small:
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

        ln, = ax.plot(median, label=label, color=color, alpha=alpha_val, linewidth=linewidth, zorder=zorder)
        ax.fill_between(
            range(len(median)), lower, upper, color=color, alpha=alpha_cloud, zorder=zorder - 1
        )
        
        lines.append(ln)
        labels.append(label)
    
    ax.set_yscale("log")
    #ax.set_ylabel("Median Test Error", fontweight="bold")
    # ax.set_title(title, loc='center', pad=10, fontsize=10, fontweight="bold")
    ax.grid(True, which="both", ls="--", lw=0.5, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    return lines, labels

for file in os.listdir("save"):
    if "active" not in file:
        continue

    #print(file)

    data = np.load(f"save/{file}")

    # Create Stacked Subplots
    fig, (ax_on, ax_off) = plt.subplots(2, 1, figsize=(4.0, 3.0), sharex=True)

    # --- Step 1: Calculate Local Limits for THIS file only ---
    # Flatten everything to find the absolute min/max for this dataset
    flat_data = np.hstack([np.array(d) for d in data.values()])
    local_ymin = np.nanmin(np.nanmedian(flat_data, axis=0)) * 0.8  # 10% buffer below
    local_ymax = np.nanmax(np.nanmedian(flat_data, axis=0)) * 1.1  # 10% buffer above

    # Then inside your loop:
    ax_on.set_ylim(local_ymin, local_ymax)
    ax_off.set_ylim(local_ymin, local_ymax)

    # Plot Top: Online
    plot_data_on_ax(ax_on, data, online_keys, "Model-Driven Planning (Online)")
    
    # Plot Bottom: Offline
    plot_data_on_ax(ax_off, data, offline_keys, "Data-Driven Selection (Offline)")
    
    # Common X-Label
    ax_off.set_xlabel("Added Samples (N)")
    ax_off.set_xticks(np.arange(10))

    # Clean Shared Legend
    # This grabs labels from both but removes duplicates
    handles, labels = ax_on.get_legend_handles_labels()
    h2, l2 = ax_off.get_legend_handles_labels()
    
    # Merge unique handles
    by_label = dict(zip(labels + l2, handles + h2))

    # 1. Define your order
    baselines = ["Passive Baseline", "Uncertainty", "Residual-Reduction"]
    results = ["Model Sensitivity", "Data Sensitivity"]

    # 2. Get the handles/labels map as before
    handles_on, labels_on = ax_on.get_legend_handles_labels()
    handles_off, labels_off = ax_off.get_legend_handles_labels()
    all_map = dict(zip(labels_on + labels_off, handles_on + handles_off))

    # 3. Separate the handles/labels into two groups
    h_top = [all_map[l] for l in baselines if l in all_map]
    l_top = [l for l in baselines if l in all_map]

    h_bot = [all_map[l] for l in results if l in all_map]
    l_bot = [l for l in results if l in all_map]

    # 4. Create the First Legend (Top Row)
    leg1 = fig.legend(
        h_top, l_top, 
        loc='lower center', 
        bbox_to_anchor=(0.5, -0.02), 
        ncol=3, 
        frameon=False, # Remove frames so they look like one unit
        columnspacing=1.5,
        handletextpad=0.4
    )

    # 5. Create the Second Legend (Bottom Row - Centered)
    leg2 = fig.legend(
        h_bot, l_bot, 
        loc='lower center', 
        bbox_to_anchor=(0.5, -0.07), # Move it slightly lower
        ncol=2, 
        frameon=False, 
        columnspacing=1.5,
        handletextpad=0.4
    )

    plt.subplots_adjust(hspace=0.15)
    fig.text(-0.02, 0.59, 'Median Test Error', va='center', rotation='vertical', fontsize=9)
        
    label_props = dict(fontweight='bold', fontsize=11, va='top', ha='left')

    # Label for the Top Plot
    ax_on.text(0.015, 1.0, '(b) Online', transform=ax_on.transAxes, fontweight='bold', va='top')
    ax_off.text(0.015, 1.0, '(c) Offline', transform=ax_off.transAxes, fontweight='bold', va='top')

    plt.tight_layout(rect=[0, 0.05, 1, 1]) # Make room for the legend at bottom
    plt.savefig(f"output/{file.split('.')[0]}.png", dpi=300, bbox_inches='tight')
    plt.close()

In [56]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Filter and sort files
files = sorted([f for f in os.listdir("save") if "ablation" in f])

# Set sharex to True for alignment, but sharey to False for independent scales
fig, axes = plt.subplots(2, 2, figsize=(5.5, 2.5), sharex=True, sharey=False)
axes = axes.flatten()

for i, file in enumerate(files[:4]):
    ax = axes[i]
    
    # 1. Load and process
    raw_data = np.load(f"save/{file}")["valid_loss"]
    x = np.arange(raw_data.shape[0])
    
    # Median and IQR calculations
    median_line = np.nanmedian(raw_data, axis=(1, 2))
    lower_bound = np.nanpercentile(raw_data, 25, axis=(1, 2))
    upper_bound = np.nanpercentile(raw_data, 75, axis=(1, 2))

    # 2. Plotting
    color = "#1f77b4"
    ax.plot(x, median_line, linewidth=1.5, color=color)
    ax.fill_between(x, lower_bound, upper_bound, color=color, alpha=0.15, lw=0)

    # 3. Panel Identifiers
    panel_label = ["(a) Tube", "(b) Slinky", "(c) Strip", "(d) Tape"]
    ax.text(0.05, 0.92, panel_label[i], transform=ax.transAxes, 
            fontsize=8, fontweight='bold', va='top', ha='left')

    # Style the individual grid and spines
    ax.grid(True, which="both", ls="--", lw=0.4, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=7)
    
    # Ensure X-ticks match your sample intervals
    ax.set_xticks(x)

# 4. The "Single Axis Label" Magic
# Create a big invisible subplot covering the whole figure
big_ax = fig.add_subplot(111, frameon=False)
# Hide its ticks and labels so it only serves as a label container
big_ax.tick_params(labelcolor='none', top=False, bottom=False, left=False, right=False)
big_ax.grid(False)

# Centered labels
big_ax.set_xlabel("Random Samples ($N$)", fontsize=9)
big_ax.set_ylabel("Median Test Error", fontsize=9)

plt.tight_layout(h_pad=0.5, w_pad=0.8) # Adjust w_pad slightly for y-axis numbers
plt.savefig("output/ablation_grid.pdf", dpi=300, bbox_inches='tight')
plt.close()

In [19]:
import numpy as np
import os

x_counts = [10, 128, 256, 512]

# Methods we care about
methods = ["MAX_S", "RANDOM"]

print(f"{'Object':<15} | {'Method':<10} | " + " | ".join([f"N={n:<8}" for n in x_counts]))
print("-" * 80)

for file in os.listdir("save"):
    if "big" not in file:
        continue

    data = np.load(f"save/{file}", allow_pickle=True)
    obj_name = file.replace(".npz", "").replace("ablation_", "").replace("big_", "").capitalize()

    for method in methods:
        if method not in data:
            continue
            
        raw_data = data[method] # Shape assumed: (len(x), num_seeds, ...) or similar
        
        # Calculate stats across all axes except the sample-count axis (axis 0)
        medians = np.nanmedian(raw_data, axis=(1, 2))
        p25 = np.nanpercentile(raw_data, 25, axis=(1, 2))
        p75 = np.nanpercentile(raw_data, 75, axis=(1, 2))
        
        # Format the row
        row_str = f"{obj_name:<15} | {method:<10}"
        for m, l, u in zip(medians, p25, p75):
            # Showing Median (IQR_lower, IQR_upper)
            row_str += f" | {m:.4f}({l:.3f},{u:.3f})"
        
        print(row_str)
    print("-" * 80)

Object          | Method     | N=10       | N=128      | N=256      | N=512     
--------------------------------------------------------------------------------
Strip           | MAX_S      | 0.0741(0.058,0.092) | 0.0644(0.054,0.077) | 0.0728(0.061,0.087) | 0.0826(0.060,0.099)
Strip           | RANDOM     | 0.0847(0.070,0.119) | 0.0896(0.074,0.115) | 0.0805(0.066,0.111) | 0.0790(0.060,0.094)
--------------------------------------------------------------------------------
Brazier         | MAX_S      | 0.0649(0.055,0.096) | 0.0678(0.055,0.109) | 0.0725(0.053,0.107) | 0.0710(0.059,0.088)
Brazier         | RANDOM     | 0.0826(0.063,0.131) | 0.0788(0.065,0.087) | 0.0705(0.063,0.089) | 0.0797(0.067,0.096)
--------------------------------------------------------------------------------
Slinky          | MAX_S      | 0.1017(0.088,0.129) | 0.1095(0.085,0.127) | 0.1083(0.088,0.130) | 0.1116(0.088,0.138)
Slinky          | RANDOM     | 0.1030(0.084,0.125) | 0.1104(0.088,0.134) | 0.1083(0.087,0.1